In [ ]:
import os
import joblib
import pandas as pd

from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
)


# ============================================================
# CONFIGURATION
# ============================================================

DATA_PATH = "data/dropout_prediction_dataset.csv"
MODEL_PATH = "model/dropout_xgboost.pkl"

TARGET = "Dropout"

# These are identifiers, NOT ML features.
ID_COLUMNS = [
    "StudentID",
    "Name",
]


# ============================================================
# 1. LOAD DATA
# ============================================================

print("Loading dataset...")

df = pd.read_csv(DATA_PATH)

print(f"Dataset shape: {df.shape}")


# ============================================================
# 2. BASIC DATA CHECKS
# ============================================================


print("\nDropout distribution:")
print(df[TARGET].value_counts())


# ============================================================
# 3. SEPARATE FEATURES AND TARGET
# ============================================================

X = df.drop(columns=ID_COLUMNS + [TARGET])
y = df[TARGET]


# ============================================================
# 4. IDENTIFY NUMERICAL AND CATEGORICAL FEATURES
# ============================================================



# ============================================================
# 5. PREPROCESSING
# ============================================================

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features,
        ),
        (
            "numerical",
            "passthrough",
            numeric_features,
        ),
    ]
)


# ============================================================
# 6. XGBOOST MODEL
# ============================================================

model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
)


# ============================================================
# 7. COMPLETE PIPELINE
# ============================================================

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model),
    ]
)


# ============================================================
# 8. TRAIN / TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)


print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))


# ============================================================
# 9. TRAIN MODEL
# ============================================================


pipeline.fit(
    X_train,
    y_train,
)



# ============================================================
# 10. PREDICTIONS
# ============================================================

y_pred = pipeline.predict(X_test)

y_probability = pipeline.predict_proba(X_test)[:, 1]


# ============================================================
# 11. EVALUATION
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred,
)

precision = precision_score(
    y_test,
    y_pred,
)

recall = recall_score(
    y_test,
    y_pred,
)

f1 = f1_score(
    y_test,
    y_pred,
)

roc_auc = roc_auc_score(
    y_test,
    y_probability,
)


print("\n" + "=" * 60)
print("MODEL PERFORMANCE")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")


print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=[
            "No Dropout",
            "Dropout",
        ],
    )
)


print("Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        y_pred,
    )
)


# ============================================================
# 12. SAVE MODEL
# ============================================================

os.makedirs(
    os.path.dirname(MODEL_PATH),
    exist_ok=True,
)

joblib.dump(
    pipeline,
    MODEL_PATH,
)

print("\nModel saved to:")
print(MODEL_PATH)


# ============================================================
# 13. TEST ONE STUDENT
# ============================================================

print("\n" + "=" * 60)
print("SAMPLE PREDICTION")
print("=" * 60)

sample_student = X_test.iloc[[0]]

actual_value = y_test.iloc[0]

prediction = pipeline.predict(
    sample_student
)[0]

probability = pipeline.predict_proba(
    sample_student
)[0][1]     


print(f"Actual Dropout: {actual_value}")
print(f"Predicted Dropout: {prediction}")
print(f"Dropout Probability: {probability * 100:.2f}%")

if prediction == 1:
    print("Risk Level: HIGH")
else:
    print("Risk Level: LOW")

Loading dataset...
Dataset shape: (500, 19)

Dropout distribution:
Dropout
0    263
1    237
Name: count, dtype: int64

Training samples: 400
Testing samples: 100

MODEL PERFORMANCE
Accuracy : 0.6700
Precision: 0.6591
Recall   : 0.6170
F1 Score : 0.6374
ROC-AUC  : 0.7258

Classification Report:
              precision    recall  f1-score   support

  No Dropout       0.68      0.72      0.70        53
     Dropout       0.66      0.62      0.64        47

    accuracy                           0.67       100
   macro avg       0.67      0.67      0.67       100
weighted avg       0.67      0.67      0.67       100

Confusion Matrix:
[[38 15]
 [18 29]]

Model saved to:
model/dropout_xgboost.pkl

SAMPLE PREDICTION
Actual Dropout: 1
Predicted Dropout: 1
Dropout Probability: 60.44%
Risk Level: HIGH
